# M3 - Maintenance prédictive des équipements lourds miniers

Notebook principal du projet **MinePredict**. Il est conçu pour une soutenance académique : il explique les choix, exécute le pipeline modulaire du backend et affiche les résultats générés à partir des datasets fournis dans `data/raw/`.

Datasets utilisés localement uniquement :
- Microsoft Azure Predictive Maintenance : telemetry, machines, failures, errors, maintenance ;
- AI4I 2020 Predictive Maintenance Dataset : `ai4i2020.csv`.

## 1. Introduction du projet

La maintenance prédictive vise à anticiper les pannes avant qu'elles n'arrêtent la production. Dans un contexte minier, les équipements lourds comme les camions-benne, foreuses et concasseurs génèrent des signaux capteurs qui peuvent révéler une dégradation progressive.

Objectifs du notebook : charger les datasets, explorer les données, construire les features temporelles, entraîner les modèles, évaluer les résultats, expliquer les prédictions et préparer les éléments visuels pour la soutenance.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Configuration du chemin - fonctionne depuis le dossier notebooks ou la racine
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

# Imports du backend
from backend.app.config.settings import get_settings
from backend.app.ml.preprocessing import load_raw_datasets, clean_dataframe, split_features_target
from backend.app.ml.feature_engineering import add_time_features, add_temporal_features, calculate_rul
from backend.app.ml.classification import train_classifiers
from backend.app.ml.rul_model import train_rul_models
from backend.app.ml.explainability import shap_summary_payload, top_feature_factors
from backend.app.services.training_service import train_all, load_metrics

# Configuration des chemins
settings = get_settings()
settings.data_raw_dir = ROOT / "data" / "raw"
settings.data_processed_dir = ROOT / "data" / "processed"
settings.model_dir = ROOT / "backend" / "saved_models"
settings.training_max_rows = 30000

RAW_DIR = settings.data_raw_dir
PROCESSED_PATH = settings.data_processed_dir / "maintenance_features.csv"
METRICS_PATH = settings.model_dir / "metrics.json"
print(f"Project root: {ROOT}")
print(f"Data directory exists: {RAW_DIR.exists()}")

## 2. Chargement des datasets

Le loader du backend fusionne proprement les tables Azure :
- `PdM_telemetry.csv` sert de table temporelle principale ;
- `PdM_machines.csv` ajoute modèle et âge ;
- `PdM_failures.csv` crée la cible `failure_binary` ;
- `PdM_errors.csv` et `PdM_maint.csv` sont agrégés en variables explicatives ;
- `ai4i2020.csv` est harmonisé dans la même table ML.

Aucun téléchargement externe n'est effectué.

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.csv"))
summary = []
for path in raw_files:
    df_tmp = pd.read_csv(path)
    summary.append({"fichier": path.name, "lignes": len(df_tmp), "colonnes": len(df_tmp.columns)})
pd.DataFrame(summary)

In [ ]:
bundle = load_raw_datasets(RAW_DIR)
print("Sources:", bundle.source_files)
print("Shape brute unifiée:", bundle.frame.shape)
print("Cible:", bundle.target_column)
print("Temps:", bundle.time_column)
print("Équipement:", bundle.equipment_column)
bundle.frame.head()

## 3. Exploration des données (EDA)

On vérifie dimensions, types, valeurs manquantes et distribution de la cible. Le très faible taux de panne est réaliste : en maintenance prédictive, les pannes sont rares et le problème est souvent déséquilibré.

In [ ]:
df = clean_dataframe(bundle.frame)
print("Dimensions après nettoyage:", df.shape)
display(df.dtypes.to_frame("type").head(40))
display(df.isna().mean().sort_values(ascending=False).head(20).to_frame("taux_na"))

In [ ]:
target = bundle.target_column
if target:
    counts = df[target].value_counts().sort_index()
    display(counts.to_frame("nombre"))
    print(f"Taux de panne: {df[target].mean():.4%}")
    counts.rename({0: "normal", 1: "panne"}).plot(kind="bar", color=["#10b981", "#ef4444"], title="Distribution panne / pas panne")
    plt.ylabel("Observations")
    plt.show()

## 4. Nettoyage des données

Le nettoyage est centralisé dans `backend/app/ml/preprocessing.py`. Cette séparation évite de mettre toute la logique métier dans le notebook et rend le projet plus professionnel.

In [ ]:
display(df.describe(include="all").T.head(30))

## 5. Analyse des corrélations

Les corrélations aident à comprendre les relations entre capteurs, mais elles ne suffisent pas à prédire les pannes rares. Elles servent surtout d'outil exploratoire.

In [ ]:
numeric_cols = [c for c in ["volt", "rotate", "pressure", "vibration", "age", "air_temperature_[k]", "process_temperature_[k]", "rotational_speed_[rpm]", "torque_[nm]", "tool_wear_[min]", target] if c in df.columns]
if numeric_cols:
    corr = df[numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Corrélations principales")
    plt.show()

## 6. Feature engineering temporel

Les features temporelles sont essentielles pour capter les tendances de dégradation : rolling mean, rolling std, lag features, tendances, moving averages et taux de changement.

In [ ]:
df_features = add_time_features(df, bundle.time_column)
df_features = add_temporal_features(df_features, bundle.equipment_column, bundle.time_column)
settings.data_processed_dir.mkdir(parents=True, exist_ok=True)
df_features.to_csv(PROCESSED_PATH, index=False)
print("Table processed:", PROCESSED_PATH)
print("Dimensions:", df_features.shape)
[c for c in df_features.columns if "rolling" in c or "lag" in c or "rate_change" in c][:25]

## 7. Visualisation temporelle capteurs

Exemple sur la machine 1 du dataset Azure. Ces courbes sont utiles en soutenance pour montrer la logique de surveillance temps réel.

In [ ]:
if "machineid" in df_features.columns:
    sample = df_features[df_features["machineid"].eq(1)].head(240)
else:
    sample = df_features.head(240)
cols = [c for c in ["volt", "rotate", "pressure", "vibration"] if c in sample.columns]
plt.figure(figsize=(12, 5))
for col in cols:
    plt.plot(sample[col].to_numpy(), label=col, linewidth=1.1)
plt.title("Aperçu télémétrie sur 240 heures")
plt.xlabel("Index temporel")
plt.legend()
plt.show()

## 8. Classification panne / pas panne

Les modèles comparés automatiquement : Logistic Regression, Random Forest, XGBoost, LightGBM si installé, SVM. La validation utilise `TimeSeriesSplit`, donc le train apprend sur le passé et le test évalue le futur.

In [ ]:
if target:
    # Pour garder le notebook rapide, on utilise le même principe que l'API : un échantillon chronologique représentatif.
    from backend.app.services.training_service import select_training_rows
    train_df = select_training_rows(df_features, settings.training_max_rows, bundle.time_column)
    X, y = split_features_target(train_df, target)
    print("Train notebook:", X.shape, "| pannes:", int(y.sum()))
    clf_result = train_classifiers(X, y)
    display(pd.DataFrame(clf_result.metrics).T)
    print("Meilleur modèle:", clf_result.best_name)

## 9. Estimation RUL

Le RUL est estimé comme le nombre d'observations restantes avant la prochaine panne connue pour un équipement. C'est une approximation pédagogique adaptée aux datasets fournis.

In [ ]:
if target:
    rul = calculate_rul(train_df, target, bundle.equipment_column, bundle.time_column)
    rul_result = train_rul_models(X, rul)
    display(pd.DataFrame(rul_result.metrics).T)
    print("Meilleur modèle RUL:", rul_result.best_name)
    pd.Series(rul).plot(kind="hist", bins=40, title="Distribution RUL calculé")
    plt.xlabel("RUL")
    plt.show()

## 10. Entraînement complet API / sauvegarde modèles

La fonction `train_all()` génère `maintenance_features.csv`, `classifier.joblib`, `rul_model.joblib` et `metrics.json`. Elle est aussi appelée par l'endpoint `POST /train`.

In [ ]:
# Décommenter pour relancer un entraînement depuis le notebook.
# result = train_all()
# result

metrics = load_metrics()
metrics

## 11. Évaluation des modèles sauvegardés

Cette section lit les métriques réelles générées par le backend pour garantir que le notebook et l'API racontent la même histoire.

In [ ]:
if "classification" in metrics:
    class_metrics = pd.DataFrame(metrics["classification"]["metrics"]).T
    display(class_metrics)
    class_metrics["f1"].plot(kind="bar", color="#26313d", title="Comparaison F1-score")
    plt.ylabel("F1-score")
    plt.show()

if "rul" in metrics:
    rul_metrics = pd.DataFrame(metrics["rul"]["metrics"]).T
    display(rul_metrics)

## 12. SHAP et explicabilité

L'objectif est d'expliquer pourquoi un modèle augmente ou réduit le risque. Pour les modèles tree-based, les importances de variables et SHAP donnent une lecture exploitable par les équipes maintenance.

In [ ]:
if target:
    factors = top_feature_factors(clf_result.best_pipeline, X.tail(5))
    display(pd.DataFrame(factors))
    shap_payload = shap_summary_payload(clf_result.best_pipeline, X.tail(100))
    display(shap_payload)

## 13. Alertes intelligentes

L'API combine la probabilité de panne, le RUL et des seuils métier sur température, vibrations et pression pour produire des alertes faible / moyen / critique.

In [ ]:
from backend.app.ml.inference import predict_failure, predict_rul, alerts_for_reading

reading = {"volt": 176, "rotate": 420, "pressure": 130, "vibration": 62, "age": 18, "temperature": 95}
print(predict_failure("TRUCK-042", reading))
print(predict_rul("TRUCK-042", reading))
display(pd.DataFrame([a.model_dump() for a in alerts_for_reading("TRUCK-042", reading)]))

## 14. Aperçus générés pour le rapport

Le script `scripts/generate_audit_report.py` produit un rapport Markdown et plusieurs figures dans `reports/figures/`. Ces éléments peuvent être repris dans le README, le rapport IMRAD et les slides.

In [ ]:
from IPython.display import Image, display
for fig in ["failure_distribution.png", "sensor_timeseries_sample.png", "correlation_heatmap.png", "model_comparison.png"]:
    path = ROOT / "reports" / "figures" / fig
    if path.exists():
        print(fig)
        display(Image(filename=str(path)))

## 15. Conclusion

Le projet tient compte des datasets réels fournis, construit une table ML unifiée, évite la fuite de données évidente, utilise une validation temporelle et expose les résultats via FastAPI + React. Les limites principales sont le déséquilibre extrême des pannes et la nature académique des datasets. Les perspectives industrielles sont l'ingestion temps réel, MLflow, monitoring de drift et base séries temporelles.